# Build Reviter on Google Colab

This notebook mirrors the Drive-backed CBCTer workflow. It mounts the authenticated
Google Drive, verifies the staged `MyDrive/Reviter` bundle, extracts the active workspace to
fast `/content` storage, runs all Pages checks, and persists the build and logs back
to Drive.

Prerequisites: access to the Drive account containing `MyDrive/Reviter`.

Outline:
1. Mount Drive and locate the staged bundle.
2. Verify its checksum and unpack it under `/content`.
3. Install dependencies and run the production validation build.
4. Inspect the persisted output summary.


## 1. Mount the persistent Drive workspace


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/Reviter')
BUILD = PROJECT / 'reviter-build'
ASSETS = PROJECT / 'assets'
OUTPUTS = PROJECT / 'reviter-outputs'
ARCHIVE = BUILD / 'reviter-source.tar.gz'
MANIFEST = BUILD / 'reviter-source-manifest.json'

assert ARCHIVE.exists(), f'Missing source bundle: {ARCHIVE}'
assert MANIFEST.exists(), f'Missing manifest: {MANIFEST}'
assert (ASSETS / 'autodesk-reference.glb').exists(), f'Missing model: {ASSETS}'
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT)
print('archive:', ARCHIVE, ARCHIVE.stat().st_size, 'bytes')


## 2. Verify and unpack into `/content`


In [ ]:
import hashlib, json, shutil, tarfile

manifest = json.loads(MANIFEST.read_text())
digest = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
assert digest == manifest['archive']['sha256'], (digest, manifest['archive']['sha256'])

workspace = Path('/content/reviter')
shutil.rmtree(workspace, ignore_errors=True)
with tarfile.open(ARCHIVE, 'r:gz') as bundle:
    destination = Path('/content').resolve()
    for member in bundle.getmembers():
        target = (destination / member.name).resolve()
        assert target == destination or destination in target.parents, member.name
    bundle.extractall(destination)

assert (workspace / 'package-lock.json').exists()
model = workspace / 'public/autodesk-reference.glb'
assert hashlib.sha256(model.read_bytes()).hexdigest() == manifest['autodeskModel']['sha256']
print('workspace:', workspace)
print('model:', model, model.stat().st_size, 'bytes')


## 3. Run the remote production build


In [ ]:
import time

run_id = time.strftime('%Y%m%d-%H%M%S')
run_output = OUTPUTS / run_id
run_output.mkdir(parents=True, exist_ok=True)
print('persistent output:', run_output)

!python /content/reviter/scripts/run_reviter_colab_build.py --source /content/reviter --output "{run_output}"


## 4. Inspect the result saved to Drive


In [ ]:
summary = json.loads((run_output / 'reviter-colab-build-summary.json').read_text())
print(json.dumps(summary, indent=2))
assert summary['ok'], summary.get('error')
print('Pages artifact:', summary['artifact']['path'])


## Optional rerun and common pitfall

To rebuild after staging a newer source bundle, rerun from **Verify and unpack**.
Do not run `npm ci` or Vite directly inside the mounted Drive directory: many small
`node_modules` writes are substantially slower there. The notebook intentionally keeps
large persistent inputs and outputs in Drive while compiling under `/content`.

Exercise: change `run_id` to a descriptive label for a comparison build. Keep each run
in a separate output directory so the prior artifact and logs remain auditable.


In [ ]:
# Answer scaffold for a named comparison run:
# run_id = 'camera-parity-comparison'
# run_output = OUTPUTS / run_id
